# Exploración del scraper — pisos.ad

Notebook para probar las funciones de `andorra_housing.scraping.base_scraper` celda a celda, viendo los resultados al momento.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from andorra_housing.scraping.base_scraper import descargar_html

In [ ]:
url = "https://www.pisos.ad/lloguer/pis-2-habitacions-1-bany-andorra-la-vella/9581"
html = descargar_html(url)
len(html), html[:300]

In [ ]:
from bs4 import BeautifulSoup
from andorra_housing.scraping.parsers.pisos_ad import extraer_campos_simples

soup = BeautifulSoup(html, "html.parser")

In [ ]:
crudos = extraer_campos_simples(soup)
crudos

In [ ]:
from andorra_housing.scraping.base_scraper import extraer_numero, extraer_parking

print("price:", extraer_numero(crudos["price_raw"]))
print("price_m2:", extraer_numero(crudos["price_m2_raw"]))
print("area_m2:", extraer_numero(crudos["area_m2_raw"]))
print("rooms:", extraer_numero(crudos["rooms_raw"]))
print("bathrooms:", extraer_numero(crudos["bathrooms_raw"]))
print("n_photos:", extraer_numero(crudos["n_photos_raw"]))
print("has_parking:", extraer_parking(crudos["parking_raw"]))

## Probar características, con un anuncio que sí tiene extras marcados

In [ ]:
from andorra_housing.scraping.parsers.pisos_ad import extraer_caracteristicas, mapear_caracteristicas

url_2 = "https://www.pisos.ad/lloguer/pis-2-habitacions-2-banys-andorra-la-vella/9454"
html_2 = descargar_html(url_2)
soup_2 = BeautifulSoup(html_2, "html.parser")

caracteristicas = extraer_caracteristicas(soup_2)
print("lista cruda:", caracteristicas)
print("booleanos:", mapear_caracteristicas(caracteristicas))

## Probar fecha de última modificación

In [ ]:
from andorra_housing.scraping.parsers.pisos_ad import extraer_fecha_modificacion

print("anuncio 1:", extraer_fecha_modificacion(soup))
print("anuncio 2:", extraer_fecha_modificacion(soup_2))

## Probar agencia y portal_id

In [ ]:
from andorra_housing.scraping.base_scraper import extraer_texto
from andorra_housing.scraping.parsers.pisos_ad import extraer_portal_id

print("agencia 1:", extraer_texto(soup, "fitxa__barraimmo-title"))
print("agencia 2:", extraer_texto(soup_2, "fitxa__barraimmo-title"))
print("portal_id 1:", extraer_portal_id(url))
print("portal_id 2:", extraer_portal_id(url_2))

## Probar la función maestra: extraer_ficha

In [ ]:
from andorra_housing.scraping.parsers.pisos_ad import extraer_ficha

ficha_1 = extraer_ficha(html, url)
ficha_2 = extraer_ficha(html_2, url_2)

ficha_1

In [ ]:
ficha_2

## Probar extracción de URLs desde una página de listado

In [ ]:
from andorra_housing.scraping.parsers.pisos_ad import extraer_urls_anuncios

with open("/tmp/salida.html", "r", encoding="utf-8") as f:
    html_listado = f.read()

urls_anuncios = extraer_urls_anuncios(html_listado)
len(urls_anuncios), urls_anuncios[:10]

## Probar el doble bucle, primero con una parroquia vacía (rápido, pocas peticiones)

In [ ]:
from andorra_housing.scraping.parsers import pisos_ad

# Sobrescritura temporal SOLO en el notebook, para probar rápido sin tocar el archivo.
pisos_ad.PARROQUIAS_AD = {1: "Andorra la Vella"}
pisos_ad.SUBTIPOS_AD = ["atic"]  # ~5 anuncios, prueba rápida y real

resultados = []
pisos_ad.recorrer_todos_los_anuncios(resultados.append)
len(resultados), resultados[0] if resultados else None

In [ ]:
for ficha in resultados:
    print(ficha["portal_id"], ficha["price"], ficha["parish"], ficha["property_type"])

## Probar el guardado del HTML crudo en data/raw/

In [ ]:
resultados = []
pisos_ad.recorrer_todos_los_anuncios(resultados.append)

from andorra_housing.scraping.base_scraper import DATA_RAW_DIR

archivos = list((DATA_RAW_DIR / "pisos_ad").glob("*.html"))
len(archivos), [f.name for f in archivos]

## Probar la base de datos: conectar y crear la tabla

In [ ]:
from andorra_housing.scraping import almacenamiento
import pandas as pd

conexion = almacenamiento.conectar()
cursor = conexion.execute("PRAGMA table_info(anuncios)")
for fila in cursor.fetchall():
    print(fila)
df = pd.read_sql("SELECT * FROM anuncios", conexion)
print(df)
df.describe()

## Ejemplo aislado de clausura, antes de probar `ejecutar()` de verdad

In [ ]:
def crear_acumulador():
    total = 0          # variable local de crear_acumulador
    vistos = []        # otra variable local

    def sumar(numero):          # función anidada: "ve" total y vistos sin que se los pasen
        nonlocal total          # dice: "el 'total' de la función de fuera, no uno nuevo"
        total += numero
        vistos.append(numero)
        print(f"sumado {numero}, total ahora: {total}, historial: {vistos}")

    return sumar   # devolvemos la función anidada, ya "cargada" con su propio total/vistos


mi_sumador = crear_acumulador()
mi_sumador(3)
mi_sumador(10)
mi_sumador(2)

In [ ]:
crear_acumulador(1) # esto da error, porque crear_acumulador no espera argumentos

## Probar `ejecutar()` de verdad, con alcance pequeño

In [ ]:
from andorra_housing.scraping.parsers import pisos_ad 
from andorra_housing.scraping.parsers.pisos_ad import ejecutar

# Reafirmamos el alcance pequeño, por si el kernel se reinició desde la última vez.
pisos_ad.PARROQUIAS_AD = {1: "Andorra la Vella"}
pisos_ad.SUBTIPOS_AD = ["atic"]

ejecutar()

In [ ]:
import pandas as pd
from andorra_housing.scraping import almacenamiento

conexion = almacenamiento.conectar()
df = pd.read_sql("SELECT * FROM anuncios", conexion)
conexion.close()

df[["portal_id", "parish", "property_type", "price", "activo", "primera_captura", "fecha_modificacion"]]

## Probar que los anuncios ya conocidos no se vuelven a descargar

In [ ]:
from andorra_housing.scraping.parsers import pisos_ad

pisos_ad.PARROQUIAS_AD = {1: "Andorra la Vella"}
pisos_ad.SUBTIPOS_AD = ["atic"]

pisos_ad.ejecutar()

## Probar la extracción LLM con una descripción real

In [ ]:
import pandas as pd
from andorra_housing.scraping import almacenamiento

conexion = almacenamiento.conectar()
df = conexion.execute("SELECT * FROM anuncios LIMIT 3").fetchdf()
display(df)
conexion.close()

In [ ]:
df.columns

In [ ]:
from andorra_housing.features.llm_extraction import extraer_de_descripcion
for i in range(len(df)):
    descripcion = df.loc[i, "descripcion_raw"]
    print(descripcion)
    print("---"*100)

    resultado, texto_crudo = extraer_de_descripcion(descripcion)

    print(resultado)
    print(texto_crudo)
    print("==="*100)

In [ ]:
import pandas as pd
from andorra_housing.scraping import almacenamiento

conexion = almacenamiento.conectar()
df = pd.read_sql("SELECT portal_id, price, area_m2, price_m2 FROM anuncios", conexion)
conexion.close()

df["price_calculado"] = (df["price_m2"] * df["area_m2"]).round()
df["diferencia"] = (df["price"] - df["price_calculado"]).abs()

df[["portal_id", "price", "area_m2", "price_m2", "price_calculado", "diferencia"]]

In [ ]:
conexion = almacenamiento.conectar()
print(conexion.execute("SHOW TABLES").fetchdf())
print(conexion.execute("DESCRIBE anuncios").fetchdf())
df = conexion.execute("SELECT * FROM anuncios").fetchdf()
conexion.close()
df.describe()


In [ ]:
for i in range(len(a)):
    print(a.loc[i].isnull)